In [6]:
import pandas as pd
import re
import ast
from sklearn.metrics import classification_report
from tqdm import tqdm

TRAIN_RAW_PATH = "../../data/raw/train.csv"

Загрузка данных

In [7]:
df_train = pd.read_csv(TRAIN_RAW_PATH, sep=";")
df_train['annotation'] = df_train['annotation'].apply(ast.literal_eval)
print(f"Загружено {len(df_train)} записей.")

df_rare = df_train[
    df_train['annotation'].apply(
        lambda ann: any(label in ['B-VOLUME', 'I-VOLUME', 'B-PERCENT', 'I-PERCENT'] for _, _, label in ann)
    )
].copy()

print(f"Найдено {len(df_rare)} записей, содержащих редкие классы 'VOLUME' или 'PERCENT'.")

Загружено 27251 записей.
Найдено 83 записей, содержащих редкие классы 'VOLUME' или 'PERCENT'.


Обработка регулярных выражений

In [8]:
num_pattern = r'(\d+[\.,]?\d*)'

volume_units = r'(л|литр|литра|литров|кг|килограмм|килограмма|г|гр|грамм|мл|миллилитров|шт|штук|уп|упак|l|ml|g|kg)'

PATTERN_VOLUME = re.compile(num_pattern + r'\s*' + volume_units, re.IGNORECASE)

percent_units = r'(%|проц|процент|процентов|жирн|жирность|ж|пр)'

PATTERN_PERCENT = re.compile(num_pattern + r'\s*' + percent_units, re.IGNORECASE)

print("Информация: Регулярные выражения успешно скомпилированы.")

Информация: Регулярные выражения успешно скомпилированы.


Функция-экстрактор

In [9]:
def extract_volume_percent(text: str) -> list:
    found_entities = []

    for match in re.finditer(PATTERN_VOLUME, text):
        start, end = match.span()
        found_entities.append((start, end, "B-VOLUME"))

    for match in re.finditer(PATTERN_PERCENT, text):
        start, end = match.span()
        found_entities.append((start, end, "B-PERCENT"))

    found_entities.sort(key=lambda e: e[1] - e[0], reverse=True)
    
    final_entities = []
    covered_indices = set()

    for start, end, label in found_entities:
        if not any(i in covered_indices for i in range(start, end)):
            final_entities.append((start, end, label))
            covered_indices.update(range(start, end))

    return sorted(final_entities)

Тестирование на обучающих данных

In [10]:
all_true_tags = []
all_pred_tags = []

for index, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Тестирование"):
    text = row["sample"]
    true_annotations = row["annotation"]
    
    true_rare_entities = [ann for ann in true_annotations if ann[2] in ('B-VOLUME', 'I-VOLUME', 'B-PERCENT', 'I-PERCENT')]
    
    tokens = text.split() 
    
    true_tags = ["O"] * len(tokens)
    for start, end, label in true_rare_entities:
        current_pos = 0
        for i, token in enumerate(tokens):
            if start <= current_pos < end:
                true_tags[i] = label
            current_pos += len(token) + 1

    pred_annotations = extract_volume_percent(text)
    pred_tags = ["O"] * len(tokens)
    for start, end, label in pred_annotations:
        current_pos = 0
        for i, token in enumerate(tokens):
            if start <= current_pos < end:
                pred_tags[i] = label
            current_pos += len(token) + 1

    all_true_tags.extend(true_tags)
    all_pred_tags.extend(pred_tags)

Тестирование: 100%|██████████| 27251/27251 [00:01<00:00, 20994.42it/s]


Результат

In [11]:
labels_to_report = ['B-VOLUME', 'I-VOLUME', 'B-PERCENT', 'I-PERCENT']
report = classification_report(all_true_tags, all_pred_tags, labels=labels_to_report, zero_division=0)

print("\n--- Отчет о качестве Regex-экстрактора на train.csv ---")
print(report)


--- Отчет о качестве Regex-экстрактора на train.csv ---
              precision    recall  f1-score   support

    B-VOLUME       0.59      0.72      0.65        57
    I-VOLUME       0.00      0.00      0.00        27
   B-PERCENT       0.76      0.50      0.60        26
   I-PERCENT       0.00      0.00      0.00         4

   micro avg       0.63      0.47      0.54       114
   macro avg       0.34      0.30      0.31       114
weighted avg       0.47      0.47      0.46       114

